In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch as tc
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

### Load Dataset

Load the ranking dataset and separate the input value columns (`val_0`–`val_9`) from the target rank columns (`rank_0`–`rank_9`). These columns are used to prepare the inputs and rank labels for the ranking task.

In [ ]:
df = pd.read_csv("ranking_dataset.csv")

value_cols = [f"val_{i}" for i in range(10)]
rank_cols = [f"rank_{i}" for i in range(10)]

### Create Dataset Class

Define a custom PyTorch dataset for the ranking task. This class converts the input values and rank labels into tensors and provides an interface for efficient data loading during training and evaluation.

In [3]:
class rank_data(Dataset):
    def __init__(self, dataframe):

        self.X = tc.tensor(dataframe[value_cols].values,dtype=tc.float32)
        self.y = tc.tensor(dataframe[rank_cols].values,dtype=tc.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

### Split Dataset

Split the dataset into training, validation, and test sets. The training set is used for learning model parameters, while the validation and test sets are used to evaluate model performance on unseen data.

In [4]:
train_df, temp_df = train_test_split(df,test_size=0.2,random_state=42)
val_df, test_df = train_test_split(temp_df,test_size=0.50,random_state=42)

train_data = rank_data(train_df)
val_data = rank_data(val_df)
test_data = rank_data(test_df)

### Create Data Loaders

Create data loaders for the training, validation, and test sets. A batch size of 64 is used, with shuffling enabled for training to improve learning and disabled for evaluation to ensure consistent results.

In [5]:
train_loader = DataLoader(train_data,batch_size=64,shuffle=True)
val_loader = DataLoader(val_data,batch_size=64,shuffle=False)
test_loader = DataLoader(test_data,batch_size=64,shuffle=False)

### MLP Baseline Model

As the first baseline experiment, a Multi-Layer Perceptron (MLP) is used to predict ranks from the input values. Hidden layer sizes of 64 and 128 features are chosen to provide sufficient learning capacity while avoiding excessive model complexity for a 10-feature input.

In [ ]:
class MLPRanker(nn.Module):

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 100)
        )

    def forward(self, x):
        out = self.net(x)
        return out.view(-1, 10, 10)

### RNN Baseline Model

As the second baseline experiment, a Recurrent Neural Network (RNN) is used to process the input values sequentially. A hidden size of 64 is chosen to maintain a compact memory representation, followed by a small classification head (64 → 32 → 10) for rank prediction. This experiment evaluates whether sequential processing provides an advantage over the MLP for the ranking task.

In [16]:
class RNNRanker(nn.Module):

    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=1,hidden_size=64,num_layers=1,batch_first=True)
        self.fc1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 10)
    def forward(self, x):
        x = x.unsqueeze(-1)
        out, _ = self.rnn(x)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)

        return out

### LSTM Baseline Model

As the third baseline experiment, an LSTM is used to process the input sequence while maintaining long-term memory through its gating mechanism. The architecture uses the same hidden size and classification head as the RNN to ensure a fair comparison, allowing the impact of the LSTM's memory capabilities on the ranking task to be evaluated.

In [17]:
class LSTMRanker(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1,hidden_size=64,num_layers=1,batch_first=True)
        self.fc1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        x = x.unsqueeze(-1)
        out, _ = self.lstm(x)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        return out

#### Evaluation Function

Define a common evaluation function for all baseline models. Accuracy is computed by comparing the predicted ranks with the true ranks across all positions, ensuring a consistent performance metric for the MLP, RNN, and LSTM experiments.

In [18]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with tc.no_grad():
        for x, y in loader:
            logits = model(x)
            preds = logits.argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

#### Training Procedure

Train each baseline model using the Adam optimizer and Cross-Entropy Loss for 20 epochs. Training and validation accuracy are monitored after every epoch to compare convergence behaviour and generalization performance across the MLP, RNN, and LSTM architectures.

In [19]:
def train_model(model, train_loader, val_loader, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = tc.optim.Adam(model.parameters(),lr=1e-3)
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for x, y in train_loader:
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits.view(-1, 10),y.view(-1)) 
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_acc = evaluate(model, train_loader)
        val_acc = evaluate(model, val_loader)
        print(
            f"Epoch {epoch+1}: "
            f"Loss={running_loss/len(train_loader):.4f} | "
            f"Train={train_acc:.4f} | "
            f"Val={val_acc:.4f}"
        )
    return model

### MLP Training and Evaluation

Train the MLP baseline for 20 epochs and evaluate its performance on the test set. The model shows steady improvement throughout training, reaching a test accuracy of around **46.84%**, indicating that it learns some ranking relationships but remains limited in capturing the global comparisons required by the task.

In [20]:
mlp_model = MLPRanker()
train_model(mlp_model,train_loader,val_loader)
mlp_test_acc = evaluate(mlp_model,test_loader)
print("MLP Test Acc:", mlp_test_acc)

Epoch 1: Loss=15.2316 | Train=0.1489 | Val=0.1504
Epoch 2: Loss=2.6488 | Train=0.1153 | Val=0.1136
Epoch 3: Loss=2.3248 | Train=0.1143 | Val=0.1088
Epoch 4: Loss=2.2988 | Train=0.1233 | Val=0.1174
Epoch 5: Loss=2.2814 | Train=0.1321 | Val=0.1287
Epoch 6: Loss=2.2675 | Train=0.1372 | Val=0.1350
Epoch 7: Loss=2.2498 | Train=0.1502 | Val=0.1445
Epoch 8: Loss=2.2047 | Train=0.1717 | Val=0.1662
Epoch 9: Loss=2.1262 | Train=0.1970 | Val=0.1929
Epoch 10: Loss=2.0291 | Train=0.2127 | Val=0.2034
Epoch 11: Loss=1.9390 | Train=0.2388 | Val=0.2312
Epoch 12: Loss=1.8617 | Train=0.2617 | Val=0.2558
Epoch 13: Loss=1.7907 | Train=0.2807 | Val=0.2828
Epoch 14: Loss=1.7354 | Train=0.2939 | Val=0.2830
Epoch 15: Loss=1.6507 | Train=0.3266 | Val=0.3148
Epoch 16: Loss=1.5876 | Train=0.3400 | Val=0.3318
Epoch 17: Loss=1.5579 | Train=0.3471 | Val=0.3384
Epoch 18: Loss=1.5384 | Train=0.3511 | Val=0.3472
Epoch 19: Loss=1.5286 | Train=0.3565 | Val=0.3501
Epoch 20: Loss=1.5168 | Train=0.3596 | Val=0.3550
MLP Test

### RNN Training and Evaluation

Train the RNN baseline for 20 epochs and evaluate its performance on the test set. The model achieves a test accuracy of around **43.08%**, showing that sequential processing alone is not sufficient for effectively solving the ranking task, despite learning some positional relationships within the sequence.

In [12]:
rnn_model = RNNRanker()
train_model(rnn_model,train_loader,val_loader)
rnn_test_acc = evaluate(rnn_model,test_loader)
print("RNN Test Acc:", rnn_test_acc)

Epoch 1: Loss=1.8789 | Train=0.3348 | Val=0.3257
Epoch 2: Loss=1.5934 | Train=0.3619 | Val=0.3551
Epoch 3: Loss=1.4784 | Train=0.3810 | Val=0.3762
Epoch 4: Loss=1.4237 | Train=0.4043 | Val=0.3990
Epoch 5: Loss=1.3957 | Train=0.4160 | Val=0.4093
Epoch 6: Loss=1.3699 | Train=0.3889 | Val=0.3876
Epoch 7: Loss=1.3783 | Train=0.4224 | Val=0.4166
Epoch 8: Loss=1.3681 | Train=0.4167 | Val=0.4090
Epoch 9: Loss=1.3570 | Train=0.4075 | Val=0.3999
Epoch 10: Loss=1.3569 | Train=0.4161 | Val=0.4122
Epoch 11: Loss=1.3474 | Train=0.4224 | Val=0.4192
Epoch 12: Loss=1.3393 | Train=0.4265 | Val=0.4264
Epoch 13: Loss=1.3445 | Train=0.4319 | Val=0.4303
Epoch 14: Loss=1.3310 | Train=0.4325 | Val=0.4339
Epoch 15: Loss=1.3265 | Train=0.4298 | Val=0.4271
Epoch 16: Loss=1.3218 | Train=0.4395 | Val=0.4350
Epoch 17: Loss=1.3226 | Train=0.4348 | Val=0.4263
Epoch 18: Loss=1.3126 | Train=0.4260 | Val=0.4263
Epoch 19: Loss=1.3079 | Train=0.4304 | Val=0.4197
Epoch 20: Loss=1.3129 | Train=0.4063 | Val=0.3984
RNN Test 

### LSTM Training and Evaluation

Train the LSTM baseline for 20 epochs and evaluate its performance on the test set. The model achieves a test accuracy of around **43.98%**, slightly outperforming the RNN, indicating that the additional memory mechanism provides a small benefit but is still insufficient for fully capturing the global comparisons required for ranking.

In [77]:
lstm_model = LSTMRanker()
train_model(lstm_model,train_loader,val_loader)
lstm_test_acc = evaluate(lstm_model,test_loader)
print("LSTM Test Acc:", lstm_test_acc)

Epoch 1: Loss=1.9706 | Train=0.3535 | Val=0.3507
Epoch 2: Loss=1.5120 | Train=0.3900 | Val=0.3890
Epoch 3: Loss=1.4215 | Train=0.4070 | Val=0.4121
Epoch 4: Loss=1.3757 | Train=0.4200 | Val=0.4177
Epoch 5: Loss=1.3499 | Train=0.4292 | Val=0.4300
Epoch 6: Loss=1.3273 | Train=0.4364 | Val=0.4313
Epoch 7: Loss=1.3209 | Train=0.4275 | Val=0.4278
Epoch 8: Loss=1.3077 | Train=0.4407 | Val=0.4316
Epoch 9: Loss=1.3057 | Train=0.4425 | Val=0.4383
Epoch 10: Loss=1.3009 | Train=0.4304 | Val=0.4253
Epoch 11: Loss=1.3062 | Train=0.4378 | Val=0.4293
Epoch 12: Loss=1.2941 | Train=0.4193 | Val=0.4125
Epoch 13: Loss=1.2862 | Train=0.4449 | Val=0.4449
Epoch 14: Loss=1.2871 | Train=0.4454 | Val=0.4443
Epoch 15: Loss=1.2826 | Train=0.4499 | Val=0.4466
Epoch 16: Loss=1.2847 | Train=0.4504 | Val=0.4498
Epoch 17: Loss=1.2844 | Train=0.4474 | Val=0.4472
Epoch 18: Loss=1.2816 | Train=0.4327 | Val=0.4284
Epoch 19: Loss=1.2758 | Train=0.4531 | Val=0.4466
Epoch 20: Loss=1.2749 | Train=0.4490 | Val=0.4392
LSTM Test

In [79]:
results = pd.DataFrame({"Model": ["MLP", "RNN", "LSTM"],"Test Accuracy": [mlp_test_acc,rnn_test_acc,lstm_test_acc]})
print(results)

  Model  Test Accuracy
0   MLP         0.4684
1   RNN         0.4308
2  LSTM         0.4398
